# P5-4. 최종 미니프로젝트 — 딥러닝 모델링 — PyTorch 코드 구현 가이드

**주제: 고객 이탈(Churn) 예측 — PyTorch DNN 직접 구현**

## 핵심 구현 흐름
`데이터 준비 → Train/Test → Train/Validation → 스케일링 → TensorDataset → DataLoader → DNN → Loss/Optimizer → 학습/Validation → EarlyStopping → Test → pos_weight → 비교`

이 노트북은 완성 코드를 제공하지 않고, P4-6과 같이 단계별 요구사항을 보고 직접 구현하도록 구성한다.

## 0. 라이브러리와 실행 장치
PyTorch, NumPy, pandas, matplotlib, scikit-learn을 불러오고 CPU/GPU 장치를 설정한다.

In [ ]:
# 필요한 라이브러리, RANDOM_STATE, device를 작성한다.

## 1. 데이터 로드
`data_save.csv`를 읽고 크기와 앞부분을 확인한다.

In [ ]:
# 데이터를 불러온다.

## 2. 전처리
범주형 컬럼을 One-Hot Encoding하고 X와 y를 분리한다.

In [ ]:
# 전처리를 수행한다.

## 3. Train/Test 분할
P5-3과 동일하게 Test 30%, stratify, random_state=42를 사용한다.

In [ ]:
# Train 전체와 Test를 분리한다.

## 4. Train/Validation 분할
Train 전체에서 Validation 20%를 분리한다. Test는 마지막 평가까지 사용하지 않는다.

In [ ]:
# Train과 Validation을 분리한다.

## 5. 스케일링
MinMaxScaler를 Train에만 fit하고 Validation/Test에는 transform만 적용한다.

In [ ]:
# 세 데이터셋을 스케일링하고 float32로 변환한다.

## 6. TensorDataset과 DataLoader
다음 조건으로 구성한다.
- X: `torch.float32`
- y: 이진분류이므로 `torch.float32`
- Train DataLoader: batch_size=32, shuffle=True
- Validation/Test: shuffle=False

In [ ]:
# TensorDataset과 DataLoader를 구현한다.

## 7. DNN 모델 정의
`nn.Module`을 상속하여 다음 구조를 만든다.

`Linear(input,64) → ReLU → Dropout(0.3) → Linear(64,32) → ReLU → Dropout(0.3) → Linear(32,1)`

**주의:** 마지막 Sigmoid는 모델 안에 넣지 않는다.

In [ ]:
# ChurnNet 클래스를 구현하고 device로 이동한다.

## 8. Loss와 Optimizer
기본 모델은 `BCEWithLogitsLoss`, Optimizer는 Adam(lr=0.001)을 사용한다.

In [ ]:
# criterion과 optimizer를 정의한다.

## 9. 학습 + Validation
직접 학습 루프를 구현한다.

Train 순서:
`model.train() → forward → loss → zero_grad → backward → step`

Validation:
`model.eval() → torch.no_grad()`

매 epoch loss와 accuracy를 저장한다.

In [ ]:
# train_model() 함수를 구현한다.

## 10. EarlyStopping
Validation loss가 좋아질 때 모델 가중치를 저장하고, 5 epoch 동안 개선이 없으면 종료한다.
학습 종료 후 가장 좋은 가중치를 다시 불러온다.

In [ ]:
# train_model() 내부에 EarlyStopping 로직을 추가한다.

## 11. 학습곡선
Train/Validation loss와 accuracy를 그래프로 비교한다.

In [ ]:
# history를 이용해 그래프를 그린다.

## 12. Test 평가 함수
예측 흐름은 다음과 같다.

`logits → torch.sigmoid() → threshold 0.5 → 0/1`

Accuracy, Precision, Recall, F1을 계산한다.

In [ ]:
# predict_binary()와 evaluate_dnn()을 구현한다.

## 13. 기본 DNN 최종 평가
학습 중 사용하지 않은 Test 데이터로 baseline 모델을 평가한다.

In [ ]:
# baseline 성능과 classification report를 출력한다.

## 14. 클래스 불균형 대응
`compute_class_weight()`로 `w0`, `w1`을 계산하고 다음 값을 만든다.

`pos_weight = w1 / w0`

새 모델의 `BCEWithLogitsLoss(pos_weight=...)`에 적용한다.

In [ ]:
# pos_weight를 계산한다.

## 15. weighted DNN 재학습 및 평가
같은 구조의 새 모델을 만들고 weighted loss로 다시 학습한다. baseline과 Test 성능을 비교한다.

In [ ]:
# weighted 모델을 학습하고 평가한다.

## 16. 최종 점검표
| 확인 항목 | 핵심 기준 |
|---|---|
| 데이터 분리 | P5-3과 동일한 Test 조건인가 |
| Validation | Train에서 별도로 만들었는가 |
| TensorDataset | X/y dtype이 적절한가 |
| DataLoader | Train만 shuffle=True인가 |
| 모델 | 최종 출력 1개 logit인가 |
| Loss | BCEWithLogitsLoss인가 |
| 학습 | zero_grad → backward → step이 있는가 |
| 평가 | eval + no_grad를 사용했는가 |
| EarlyStopping | best weight를 복원했는가 |
| 불균형 | pos_weight 전후를 비교했는가 |
| 최종 평가 | Accuracy, Precision, Recall, F1을 비교했는가 |